In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns



db_path = r"../data/database/faers_2025.db"
conn = sqlite3.connect(db_path)




In [ ]:
# Load the cleaned DEMO table
df_demo = pd.read_sql_query("SELECT * FROM demo_clean", conn)

In [ ]:
# ---------------------------------------------------------
# 1. General Overview & Null Check
# ---------------------------------------------------------
print(" 1. Data Types and Missing Values Check:")
# Instead of info(), this gives a cleaner look at missing values in the cleaned dataset
missing_df = df_demo.isnull().sum().to_frame(name='Missing Count')
missing_df['% Missing'] = (missing_df['Missing Count'] / len(df_demo)) * 100
print(missing_df.round(2))
print("-" * 50)

# ---------------------------------------------------------
# 2. Numerical Summary (Age & Weight Sanity Check)
# ---------------------------------------------------------
print("\n 2. Numerical Summary (Age & Weight):")
# We expect age to be strictly in Years, and wt strictly in KG
print(df_demo[['age', 'wt']].describe().round(2))
print("-" * 50)

# ---------------------------------------------------------
# 3. Categorical Distributions
# ---------------------------------------------------------
print("\n 3. Categorical Value Counts:")
print("SEX Distribution:\n", df_demo['sex'].value_counts(dropna=False))
print("\nREPORT TYPE (I_F_CODE):\n", df_demo['i_f_code'].value_counts(dropna=False))
print("-" * 50)

# ---------------------------------------------------------
# 4. Visualizing the Cleaned Distributions
# ---------------------------------------------------------
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# Plot Age Distribution
sns.histplot(df_demo['age'].dropna(), bins=50, kde=True, ax=ax[0], color='royalblue')
ax[0].set_title('Age Distribution (Strictly Years)')
ax[0].set_xlabel('Age')
ax[0].set_ylabel('Frequency')

# Plot Weight Distribution
sns.histplot(df_demo['wt'].dropna(), bins=50, kde=True, ax=ax[1], color='seagreen')
ax[1].set_title('Weight Distribution (Strictly KG)')
ax[1].set_xlabel('Weight (KG)')
ax[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

# Free memory
del df_demo


In [ ]:
df_drug = pd.read_sql_query("SELECT role_cod, dechal, rechal FROM drug_clean", conn)

print(df_drug['role_cod'].value_counts(dropna=False))

print("\nDECHAL Distribution (Y = Positive, N = Negative, UNK = Unknown):")
print(df_drug['dechal'].value_counts(dropna=False))
del df_drug

# Check dimensionality reduction success on Drug Names
df_drug_names = pd.read_sql_query("""
SELECT final_drug_name, COUNT(*) as frequency 
FROM drug_clean 
GROUP BY final_drug_name 
ORDER BY frequency DESC 
LIMIT 5
""", conn)
print("\nTop 5 Standardized Drug Names (final_drug_name):")
print(df_drug_names)

print("-" * 50)

In [ ]:
df_ther = pd.read_sql_query("SELECT dur, dur_cod FROM ther_clean WHERE dur IS NOT NULL", conn)

print("Duration (in DAYS) Summary Statistics:")
print(df_ther['dur'].describe().round(2))

# Visualize the duration distribution (using a log scale since medical durations vary wildly)
plt.figure(figsize=(8, 5))
sns.histplot(df_ther['dur'], bins=50, kde=False, color='darkorange', log_scale=True)
plt.title('Distribution of Therapy Duration in Days (Log Scale)')
plt.xlabel('Duration (Days) - Log Scale')
plt.ylabel('Frequency')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

del df_ther
print("-" * 50)

In [ ]:
# 3. Target Variables Validation (REAC & OUTC)
# ---------------------------------------------------------
print("🎯 3. Target Variables (REAC & OUTC) Validation:")

# REAC: Most frequent adverse events
df_reac = pd.read_sql_query("""
SELECT pt, COUNT(*) as event_count 
FROM reac_clean 
GROUP BY pt 
ORDER BY event_count DESC 
LIMIT 5
""", conn)
print("Top 5 Adverse Events (Preferred Terms):")
print(df_reac)

# OUTC: Outcome code distribution
df_outc = pd.read_sql_query("""
SELECT outc_cod, COUNT(*) as outcome_count 
FROM outc_clean 
GROUP BY outc_cod 
ORDER BY outcome_count DESC
""", conn)
print("\nOutcome Codes Distribution:")
print(df_outc)

# Clean up
conn.close()